# Gigs Senior Data Analyst Challenge

Welcome to the Gigs data analyst take-home challenge! This notebook will help you get started with analyzing our connectivity usage data.

## About the Data

You'll be working with three main datasets:
- **Usage Data**: Detailed usage per subscription period (~100K+ records)
- **Plan Events**: Plan configuration and pricing history
- **Projects**: Project metadata

## Setup Instructions

Run the cells below to set up your environment and load the data into DuckDB.

In [96]:
# Import required libraries
import duckdb
import pandas as pd
from datetime import datetime, timedelta

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [97]:
# Load JupySQL extension and configure
%load_ext sql

# Configure JupySQL for better output
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

print("✅ JupySQL configured!")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
✅ JupySQL configured!


In [98]:
# Connect to DuckDB
conn = duckdb.connect('gigs-analytics.db')
%sql conn --alias duckdb

print("✅ Connected to DuckDB database: gigs-analytics.db")

✅ Connected to DuckDB database: gigs-analytics.db


In [99]:
%%sql
-- Load data into DuckDB tables
CREATE OR REPLACE TABLE usage_data AS 
SELECT * FROM 'data/usage_by_subscription_period.csv';

CREATE OR REPLACE TABLE plan_events AS 
SELECT * FROM 'data/plan_change_events.csv';

CREATE OR REPLACE TABLE projects AS 
SELECT * FROM 'data/projects.csv';

,Count
0,3


In [100]:
%%sql
-- Verify data loading
select 
  'usage_data' as table_name, 
  count(*) as row_count,
  count(distinct subscription_id) as unique_subscriptions
from usage_data
union all
select 
  'plan_events' as table_name, 
  count(*) as row_count,
  count(distinct plan_id) as unique_plans
from plan_events
union all
select 
  'projects' as table_name, 
  count(*) as row_count,
  count(distinct project_id__hashed) as unique_projects
from projects;

,table_name,row_count,unique_subscriptions
0,usage_data,53565,8457
1,plan_events,209,36
2,projects,3,3


## Your Analysis Starts Here!

Now you have everything set up. Use the cells below to start your analysis.

### Tips:
- Use `%%sql` for multi-line SQL queries
- Use `%sql variable_name <<` to store results in a Python variable
- Combine SQL with Python/Pandas for advanced analysis
- Feel free to use any visualisation library you feel comfortable with

---
---
# Start

In [101]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px


In [102]:
#to check : usage behaviour can vary significantly across customer types and verticals.

# tables 

# usage_data
# plan_events
# projects

In [103]:
pro = pd.read_csv('data/projects.csv')
df = pd.read_csv('data/usage_by_subscription_period.csv')
plan = pd.read_csv('data/plan_change_events.csv')

In [104]:
pro.head()

,project_id__hashed,project_type,organization_name,device_type
0,dace2786aee7632e61757b320a6fe5bff37a2e742fe558...,API,People Mobile,Phones
1,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,Connect,ACME Phone,Phones
2,2aeca1a6c1ecf52b28b7f7646b6fb90563a417ee3e5dc3...,Connect,SmartDevices Inc.,Wearables


In [105]:
plan.head(3)


,plan_id,project_id__hashed,plan_created_at,event_type,event_timestamp,plan_name,network_provider_id,price_currency,plan_price_amount_local,data_allowance_mb,is_unlimited_data,voice_allowance_seconds,is_unlimited_voice,sms_allowance,is_unlimited_sms,validity_value,validity_unit,_valid_from,_valid_to,_is_current_state
0,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2023-12-29 18:44:37.573698,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:44:37.573698,2023-12-29 18:44:37.846803,False
1,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.archived,2024-01-21 15:48:13.444971,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:13.444971,NaN,True
2,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2024-01-21 15:48:12.953812,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:12.953812,2024-01-21 15:48:13.444971,False


In [106]:
plan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   plan_id                  209 non-null    object 
 1   project_id__hashed       209 non-null    object 
 2   plan_created_at          209 non-null    object 
 3   event_type               209 non-null    object 
 4   event_timestamp          209 non-null    object 
 5   plan_name                209 non-null    object 
 6   network_provider_id      209 non-null    object 
 7   price_currency           209 non-null    object 
 8   plan_price_amount_local  209 non-null    float64
 9   data_allowance_mb        120 non-null    float64
 10  is_unlimited_data        209 non-null    bool   
 11  voice_allowance_seconds  41 non-null     float64
 12  is_unlimited_voice       209 non-null    bool   
 13  sms_allowance            35 non-null     float64
 14  is_unlimited_sms         2

## DATA QUALITY CHECKS



In [92]:
def show_stats(df):
    for cols in df.columns:
        print(f"{cols} {'-'*(20-len(cols))}:  unique[{df[cols].nunique()}], null: {df[cols].isnull().sum()}, na:{df[cols].isna().sum()} \n samples --> {df[cols].unique()[:3]}", end="\n\n")

print('-----------------------------------------')
print('---------------- project ----------------')
print('-----------------------------------------\n\n')
show_stats(pro)
print('-----------------------------------------')
print('-----------------subscriptions-----------')
print('-----------------------------------------\n\n')
show_stats(df)
print('----------------------------------------------')
print('-----------------plan_change_events-----------')
print('----------------------------------------------\n\n')
show_stats(plan)







-----------------------------------------
---------------- project ----------------
-----------------------------------------


project_id__hashed --:  unique[3], null: 0, na:0 
 samples --> ['dace2786aee7632e61757b320a6fe5bff37a2e742fe5582c245665e3d0e3a84e'
 '82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f3597cb2817a9b5a87bbf3'
 '2aeca1a6c1ecf52b28b7f7646b6fb90563a417ee3e5dc32f9d1a50d7e8b9e13a']

project_type --------:  unique[2], null: 0, na:0 
 samples --> ['API' 'Connect']

organization_name ---:  unique[3], null: 0, na:0 
 samples --> ['People Mobile' 'ACME Phone' 'SmartDevices Inc.']

device_type ---------:  unique[2], null: 0, na:0 
 samples --> ['Phones' 'Wearables']

-----------------------------------------
-----------------subscriptions-----------
-----------------------------------------


subscription_id -----:  unique[8457], null: 0, na:0 
 samples --> ['sub_b97107a1c7ef89cf28a7e83ee850' 'sub_2353ef9e8bf55916b97ceb30194c'
 'sub_5d5605c8cf0b46116a42cedb9561']

project_id__hash

In [94]:
plan.event_type.unique()

array(['plan.updated', 'plan.archived', 'plan.published', 'plan.created'],
      dtype=object)

## Observations:
  
Project: clean

Subscription: 
1. subscription start and end have different lenght, to check why.
1. cumulative data usage can be 0 
1. same for sms
  
Plan: 
1. voice_allowance_seconds , sms_allowance  could need some cleaning, but here we wont use it
1. plan archived could be use for churn



In [61]:
%%sql

# can a subscription have multiple plans? - YES

SELECT 

  subscription_id,
  count(distinct plan_id) as total_plans
FROM usage_data
group by 1
order by 2 desc 
limit 2; 

,subscription_id,total_plans
0,sub_bf65d729c2df4092017eaea0ee0a,3
1,sub_7527d51ab0c81f907d51aa3f5ee2,3


In [ ]:
%%sql

# example of a subscription with multiple plans
select 
* 
from usage_data

where subscription_id = 'sub_bf65d729c2df4092017eaea0ee0a';



,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-05-24,2024-04-25,2024-05-25,1,1829.331968,773.0,214.0,0
1,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-06-24,2024-05-25,2024-06-25,2,1488.722944,1761.0,484.0,0
2,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-07-24,2024-06-25,2024-07-25,3,1360.831488,1695.0,1205.0,0
3,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-08-24,2024-07-25,2024-08-25,4,1040.356352,1370.0,340.0,0
4,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-09-24,2024-08-25,2024-09-25,5,1449.269248,1179.0,243.0,0
5,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-10-24,2024-09-25,2024-10-25,6,1337.875456,1093.0,289.0,0
6,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-11-24,2024-10-25,2024-11-25,7,1284.484096,1306.0,443.0,0
7,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_af7b4a30086929d37e7f4eb3daed,2024-12-24,2024-11-25,2024-12-25,8,9843.621888,1300.0,553.0,0
8,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2025-01-24,2024-12-25,2025-01-25,9,13642.873856,1021.0,229.0,0
9,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2025-02-24,2025-01-25,2025-02-25,10,19221.571584,706.0,188.0,0


In [ ]:
%%sql

# testing if overlapping periods are possible- YES

select 
subscription_id,
max(subscription_period_number) as max_subscription_period_number,
count(distinct subscription_period_number) as total_subscription_period_number
from usage_data
group by 1
having max_subscription_period_number != total_subscription_period_number;

,subscription_id,max_subscription_period_number,total_subscription_period_number
0,sub_94e55518b8a9d8e6c18108de13a6,14,8
1,sub_b05686892dcc5e429cde4571596c,7,4
2,sub_33f7eb034d22a8eabce2bceecd6f,14,8
3,sub_d4120772c9481e173cfb9d56c15a,16,15
4,sub_43cf31d5a76b5c54ea600a770d70,13,12
...,...,...,...
125,sub_01c7593ff53bcf2e90678637e0cd,3,2
126,sub_b2aaaa8ae16e8e5493e2d7317103,5,4
127,sub_552e85f62a1fb73905fe369df752,7,5
128,sub_a86663275ca75dd240589046e1cc,12,7


In [ ]:
%%sql 

# -- there are overlapping period_numbers  as well as missing periods 

select 
* 
from usage_data

where subscription_id = 'sub_b05686892dcc5e429cde4571596c';





,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-03-29,2024-02-29,2024-03-30,1,342.447104,144.0,172.0,0
1,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-05-30,2024-04-30,2024-05-30,3,0.000000,0.0,0.0,0
2,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-08-29,2024-07-31,2024-08-29,6,0.000000,0.0,0.0,0
3,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-09-30,2024-08-29,2024-09-30,7,0.000000,0.0,0.0,0
4,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-10-21,2024-09-30,2024-10-21,7,0.000000,0.0,0.0,0


In [109]:
%%sql 

select 
report_date,subscription_period_start,subscription_period_end,
date_diff(subscription_period_end,subscription_period_start) as subscription_period_duration

from usage_data

where subscription_id = 'sub_b05686892dcc5e429cde4571596c';





BinderException: Binder Error: Referenced column "report_date" not found in FROM clause!
Candidate bindings: "reporting_date", "project_id__hashed", "plan_id", "subscription_period_start", "subscription_period_number"

LINE 2: report_date,subscription_period_start,subscription_period_end...
        ^

In [ ]:

%%sql
# can a subscription have multiple projects? - NO

SELECT 
subscription_id,
count(distinct project_id__hashed) as total_projects
FROM usage_data
group by 1

order by 2 desc 
limit 3; 

,subscription_id,total_projects
0,sub_ca64333ea71eccafa2b4486ee638,1
1,sub_096328a7111b9c85611b8a9b563a,1
2,sub_a3892b48c50d9ee670213a35b78c,1


In [15]:
# Q1 : How much data does a subscription typically consume?


In [ ]:
%%sql
SELECT 

 *
FROM usage_data;


,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_b97107a1c7ef89cf28a7e83ee850,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-07,2024-02-04,2024-02-07,1,0.000000,0.00,0.0,0
1,sub_2353ef9e8bf55916b97ceb30194c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-08,2024-01-25,2024-02-08,1,56.660992,1.00,1.0,0
2,sub_5d5605c8cf0b46116a42cedb9561,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2024-02-08,2024-01-25,2024-02-08,1,160.659456,0.00,1.0,0
3,sub_8e114d3a638d12ca1526a3fbedd6,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-08,2024-01-28,2024-02-08,1,156.870656,93.00,198.0,0
4,sub_16059484727214a6b903fa16d0fe,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-10,2024-02-09,2024-02-10,1,0.000000,0.00,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...
53560,sub_77c957122b1c0b43cf8fa67a00ed,dace2786aee7632e61757b320a6fe5bff37a2e742fe558...,pln_b109c69e95246b6d535b9cc9174f,2025-06-16,2025-05-18,2025-06-17,1,42.014720,11.44,22.0,0
53561,sub_2a18ead2e6896d98692a1ac1a974,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2025-06-16,2025-05-17,2025-06-17,10,0.000000,0.00,0.0,0
53562,sub_4d5db9c62fd12ab579e5ce11976f,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2025-06-16,2025-05-17,2025-06-17,11,60.177408,802.00,860.0,0
53563,sub_21440dbd85be68b35f7103e181d1,dace2786aee7632e61757b320a6fe5bff37a2e742fe558...,pln_3547586bf1a44e88d28081a6a456,2025-06-16,2025-05-18,2025-06-17,1,4952.474624,264.01,146.0,0


In [29]:
# can a subscription have multiple projects?
%%sql
SELECT 

  subscription_id,
  sum(cumulative_data_usage_megabyte) as total_cumulative_data_usage_megabyte,
  avg(cumulative_data_usage_megabyte) as avg_cumulative_data_usage_megabyte

FROM usage_data

group by 1

;



IndentationError: unexpected indent (387840675.py, line 5)